# 10 — Compile Academic Report

This notebook compiles the pipeline outputs into a publication-style report (Markdown + PDF) with provenance and validation appendices.

Outputs:
- `outputs/10_report/report.md`
- `outputs/10_report/report.pdf`
- `outputs/10_report/figures/*.png`
- `outputs/10_report/tables/*.csv`
- `outputs/10_report/manifest.json`


In [ ]:
import os, json, hashlib
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import matplotlib.pyplot as plt

# reportlab is optional locally; in GitHub Actions add it to requirements.txt
from reportlab.lib.pagesizes import A4
from reportlab.lib.units import cm
from reportlab.pdfgen import canvas

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def sha256_file(p: Path) -> str:
    return sha256_bytes(p.read_bytes())

def write_json(path: Path, obj) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2), encoding="utf-8")

def read_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(12):
        if (cur/"configs").exists() and (cur/"outputs").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    return start.resolve()

ROOT = find_repo_root(Path.cwd())
OUTPUTS = ROOT/"outputs"
CONFIGS = ROOT/"configs"

print("ROOT:", ROOT)
print("Outputs exists:", OUTPUTS.exists())

In [ ]:
# Load compiled manifests if present
compiled = OUTPUTS/"09_compile_step_manifest/all_step_manifests.json"
manifests = []
if compiled.exists():
    manifests = read_json(compiled)
else:
    # fallback: scan outputs/*/manifest.json
    for p in sorted(OUTPUTS.glob("*/manifest.json")):
        manifests.append(read_json(p))

mf_rows=[]
for m in manifests:
    mf_rows.append({
        "step": m.get("step"),
        "created_at_utc": m.get("created_at_utc"),
        "n_requests": len(m.get("requests",[])),
        "n_artifacts": len(m.get("artifacts",[])),
    })
mf = pd.DataFrame(mf_rows).sort_values("step")
mf

In [ ]:
# Basic validation checks (non-blocking, but recorded)
checks = []

def check_exists(path: Path, label: str):
    ok = path.exists()
    checks.append({"check": label, "path": str(path), "ok": ok})
    return ok

check_exists(CONFIGS/"run.yml", "run_config_present")
check_exists(OUTPUTS/"05_metoffice_datahub_weather/derived/nearest_index.json", "metoffice_nearest_index_present")
check_exists(OUTPUTS/"04_ground_aq_providers/provider_calls.json", "ground_provider_calls_present")

pd.DataFrame(checks)

In [ ]:
# Build report assets
STEP = "10_report"
out = OUTPUTS/STEP
figdir = out/"figures"
tabdir = out/"tables"
out.mkdir(parents=True, exist_ok=True)
figdir.mkdir(parents=True, exist_ok=True)
tabdir.mkdir(parents=True, exist_ok=True)

# 1) Manifest overview table
mf.to_csv(tabdir/"step_manifest_overview.csv", index=False)

# 2) Provider call status summary if present
prov_path = OUTPUTS/"04_ground_aq_providers/provider_calls.json"
prov_df = None
if prov_path.exists():
    prov = read_json(prov_path)
    prov_df = pd.DataFrame(prov)
    if not prov_df.empty and "ok" in prov_df.columns:
        prov_summary = (prov_df
            .assign(ok=prov_df["ok"].astype(bool))
            .groupby(["provider","ok"], dropna=False)
            .size().reset_index(name="count"))
        prov_summary.to_csv(tabdir/"ground_provider_call_summary.csv", index=False)

# 3) Met Office station mapping table
nearest_path = OUTPUTS/"05_metoffice_datahub_weather/derived/nearest_index.json"
nearest_df = None
if nearest_path.exists():
    nearest_df = pd.DataFrame(read_json(nearest_path))
    nearest_df.to_csv(tabdir/"metoffice_nearest_index.csv", index=False)

# 4) Quick plot: counts of artifacts per step
plt.figure()
mf.plot(x="step", y="n_artifacts", kind="bar", legend=False)
plt.tight_layout()
plt.savefig(figdir/"artifacts_per_step.png", dpi=200)
plt.close()

# 5) Quick plot: provider success/failure counts
if prov_df is not None and not prov_df.empty and "provider" in prov_df.columns and "ok" in prov_df.columns:
    plt.figure()
    (prov_df.assign(ok=prov_df["ok"].astype(bool))
           .groupby(["provider","ok"]).size()
           .unstack(fill_value=0)
           .plot(kind="bar"))
    plt.tight_layout()
    plt.savefig(figdir/"ground_provider_ok_counts.png", dpi=200)
    plt.close()

print("Wrote tables to", tabdir)
print("Wrote figures to", figdir)

In [ ]:
# Compose Markdown report (academic-style skeleton)
run_cfg_text = (CONFIGS/"run.yml").read_text(encoding="utf-8") if (CONFIGS/"run.yml").exists() else ""

# Compose Markdown report (academic-style skeleton)
run_cfg_text = (CONFIGS/"run.yml").read_text(encoding="utf-8") if (CONFIGS/"run.yml").exists() else ""

report_md = []
report_md.append("# AQ26 Environmental Forensic Assessment Report")
report_md.append("")
report_md.append(f"**Generated (UTC):** {utc_now()}")
report_md.append("")
report_md.append("## 1. Introduction")
report_md.append("This report compiles outputs from the AQ26 pipeline to support an environmental forensic assessment. It integrates meteorology, ground air-quality indicators, and contextual evidence (news) where available. The workflow is provenance-first: raw responses are retained, hashes are recorded, and each step emits a manifest.")
report_md.append("")
report_md.append("## 2. Study design")
report_md.append("### 2.1 Sites and roles")
if nearest_df is not None and not nearest_df.empty:
    report_md.append("The Met Office Land Observations API was queried for the nearest observing station for each configured site, using the `/nearest` endpoint followed by observations using `/{geohash}`.")
report_md.append("")
report_md.append("### 2.2 Data sources")
report_md.append("- Met Office Weather DataHub (Land Observations)")
report_md.append("- Ground AQ providers (WAQI / IQAir / Ambee) when configured")
report_md.append("- UK-AIR catalogue discovery when enabled")
report_md.append("- Sentinel-5P acquisition planning (CDSE) when enabled")
report_md.append("")
report_md.append("## 3. Methods")
report_md.append("### 3.1 Provenance and integrity controls")
report_md.append("Each pipeline step writes a `manifest.json` containing: (a) input config hashes, (b) request metadata, (c) output artifact hashes. This allows independent verification that outputs match the recorded inputs and API responses.")
report_md.append("")
report_md.append("### 3.2 Validation checks")
report_md.append("The following basic checks were run during report compilation:")
report_md.append("")
chk_df = pd.DataFrame(checks)
report_md.append(chk_df.to_markdown(index=False))
report_md.append("")
report_md.append("## 4. Results")
report_md.append("### 4.1 Pipeline execution summary")
report_md.append("Artifacts per step:")
report_md.append("")
report_md.append("![](figures/artifacts_per_step.png)")
report_md.append("")
if prov_df is not None and not prov_df.empty:
    report_md.append("### 4.2 Ground AQ provider calls")
    if (tabdir/"ground_provider_call_summary.csv").exists():
        report_md.append("Call success/failure summary by provider is provided in the appendices.")
    if (figdir/"ground_provider_ok_counts.png").exists():
        report_md.append("")
        report_md.append("![](figures/ground_provider_ok_counts.png)")
        report_md.append("")
report_md.append("## 5. Interpretation")
report_md.append("Interpretation should be constrained by the evidential limits of each data source. Land Observations represent station conditions (not stack emissions), and regional satellite columns are not direct proxies for local source strength. Control-site comparisons must be conditioned on wind sector and stability.")
report_md.append("")
report_md.append("## 6. Conclusions")
report_md.append("- [To be completed after adding the statistical comparison notebook.]")
report_md.append("")
report_md.append("## 7. Recommendations")
report_md.append("- Add wind-sector filtering and upwind/downwind tagging per hour.")
report_md.append("- Implement retry/backoff and provider-specific rate limiting.")
report_md.append("- Extend `run.yml` to include explicit `news`, `satellite`, and `report` sections.")
report_md.append("")
report_md.append("## Appendix A — Configuration (run.yml)")
report_md.append("```yaml")
report_md.append(run_cfg_text.strip())
report_md.append("```")
report_md.append("")
report_md.append("## Appendix B — Step manifests and artifacts")
report_md.append("A machine-readable overview table is in `tables/step_manifest_overview.csv`.")
report_md.append("")
if (tabdir/"metoffice_nearest_index.csv").exists():
    report_md.append("## Appendix C — Met Office nearest station mapping")
    report_md.append("Saved to `tables/metoffice_nearest_index.csv`.")
report_md.append("")

(out/"report.md").write_text("\n".join(report_md), encoding="utf-8")
print("Wrote:", out / "report.md")

In [ ]:
# Render a simple PDF from the Markdown content (keeps dependencies light)
# This PDF is intentionally conservative: title page + key figures + appendix pointers.
pdf_path = out/"report.pdf"
c = canvas.Canvas(str(pdf_path), pagesize=A4)
w, h = A4

def draw_wrapped(text, x, y, max_width_chars=95, line_height=14):
    import textwrap as _tw
    for line in _tw.wrap(text, width=max_width_chars):
        c.drawString(x, y, line)
        y -= line_height
    return y

y = h - 2*cm
c.setFont("Helvetica-Bold", 16)
c.drawString(2*cm, y, "AQ26 Environmental Forensic Assessment Report")
y -= 1.2*cm
c.setFont("Helvetica", 10)
y = draw_wrapped(f"Generated (UTC): {utc_now()}", 2*cm, y, 110, 12)
y -= 0.5*cm

c.setFont("Helvetica-Bold", 12)
c.drawString(2*cm, y, "Pipeline execution summary")
y -= 0.8*cm
c.setFont("Helvetica", 10)
y = draw_wrapped("This PDF is a compact companion to report.md. See the Markdown for full tables and appendices.", 2*cm, y)
y -= 0.5*cm

# Embed figure if present
fig1 = figdir/"artifacts_per_step.png"
if fig1.exists():
    c.drawImage(str(fig1), 2*cm, y-8*cm, width=17*cm, height=8*cm, preserveAspectRatio=True, mask='auto')
    y -= 9*cm

fig2 = figdir/"ground_provider_ok_counts.png"
if fig2.exists():
    if y < 10*cm:
        c.showPage()
        y = h - 2*cm
    c.setFont("Helvetica-Bold", 12)
    c.drawString(2*cm, y, "Ground provider call status")
    y -= 0.8*cm
    c.drawImage(str(fig2), 2*cm, y-8*cm, width=17*cm, height=8*cm, preserveAspectRatio=True, mask='auto')
    y -= 9*cm

c.showPage()
c.setFont("Helvetica-Bold", 12)
c.drawString(2*cm, h-2*cm, "Appendices and provenance")
c.setFont("Helvetica", 10)
draw_wrapped("See outputs/10_report/report.md plus the per-step manifests in outputs/<step>/manifest.json for full provenance, including SHA256 hashes of inputs/outputs.", 2*cm, h-3*cm)

c.save()
print("Wrote:", pdf_path)